In [1]:
import numpy as np
import pandas as pd

from sklearn.feature_selection import mutual_info_regression
from sklearn.ensemble import RandomForestRegressor

In [2]:
# Cargar dataset
df = pd.read_csv('dataset.csv')

print(f"Dataset cargado correctamente: {df.shape[0]} filas, {df.shape[1]} columnas")

cols_sensores = [c for c in df.columns if not c.startswith('A_')]
cols_acciones = [c for c in df.columns if c.startswith('A_')]

X = df[cols_sensores]
y = df[cols_acciones]

# Valores constantes
std = X.std()
constantes = std[std == 0].index.tolist()
if constantes:
    print(f"\nEliminados {len(constantes)} sensores con valores constantes.")
    X = X.drop(columns=constantes)

# Valores redundantes
umbral = 0.95

cols_para_correlacion = [
    c for c in X.columns
    if not c.startswith('track_') and not c.startswith('focus_')
]

correlaciones = X[cols_para_correlacion].corr().abs()

superior = correlaciones.where(
    np.triu(np.ones(correlaciones.shape), k=1).astype(bool)
)

eliminados = [
    column for column in superior.columns
    if any(superior[column] > umbral)
]

if eliminados:
    print(f"Eliminados {len(eliminados)} sensores con valores redundantes.")
    X = X.drop(columns = eliminados)
else:
    print("No se han encontrado sensores con valores redundantes.")

# Filas duplicadas
df_limpio = pd.concat([X, y], axis=1)

df_final = df_limpio.drop_duplicates()

instancias = df.shape[0] - df_final.shape[0]

if instancias > 0:
    print(f"Eliminadas {instancias} filas duplicadas.")
else:
    print("No se han encontrado filas duplicadas.")

# Guardar dataset limpio
df_final.to_csv('dataset_limpio.csv', index=False)

print(f"\nDataset guardado correctamente: {df_final.shape[0]} filas, {df_final.shape[1]} columnas")

print(f"\nSensores guardados: {list(X.columns)}")

Dataset cargado correctamente: 101089 filas, 38 columnas

Eliminados 5 sensores con valores constantes.
Eliminados 4 sensores con valores redundantes.
Eliminadas 60 filas duplicadas.

Dataset guardado correctamente: 101029 filas, 29 columnas

Sensores guardados: ['angle', 'track_pos', 'z', 'speed', 'rpm', 'gear', 'track_0', 'track_1', 'track_2', 'track_3', 'track_4', 'track_5', 'track_6', 'track_7', 'track_8', 'track_9', 'track_10', 'track_11', 'track_12', 'track_13', 'track_14', 'track_15', 'track_16', 'track_17', 'track_18']


In [5]:
# Cargar dataset
df = pd.read_csv('dataset_limpio.csv')
    
cols_sensores = [c for c in df.columns if not c.startswith('A_')]

X = df[cols_sensores]
    
mean = X.mean(axis=0)
std = X.std(axis=0)
    
print(f"{'SENSOR':<15} {'MEAN':<15} {'STD':<15}")
print("-" * 40)

for col in cols_sensores:
    print(f"{col:<15} {mean[col]:<15.2f} {std[col]:<8.2f}")

print("\nTotal sensores:", len(cols_sensores))

SENSOR          MEAN            STD            
----------------------------------------
angle           0.00            0.08    
track_pos       -0.03           0.40    
z               0.34            0.01    
speed           87.83           39.11   
rpm             4476.60         912.54  
gear            3.08            1.20    
track_0         8.82            5.90    
track_1         9.19            6.17    
track_2         10.43           7.11    
track_3         13.94           12.30   
track_4         32.63           39.48   
track_5         42.44           41.03   
track_6         44.91           40.66   
track_7         48.42           41.86   
track_8         54.01           45.86   
track_9         60.56           52.48   
track_10        53.75           49.29   
track_11        44.00           40.35   
track_12        36.01           31.12   
track_13        29.80           26.56   
track_14        16.87           11.90   
track_15        11.18           7.78    
track_16 

In [3]:
# Cargar dataset
df = pd.read_csv('dataset_limpio.csv')

cols_sensores = [c for c in df.columns if not c.startswith('A_')]
cols_acciones = [c for c in df.columns if c.startswith('A_')]

X = df[cols_sensores]
y = df[cols_acciones]

indices = X.index.intersection(y.index)
X = X.loc[indices]
y = y.loc[indices]
print(f"Datos sincronizados: {len(X)} filas analizadas.")

def normalizar(series):
    rango = series.max() - series.min()
    if rango == 0:
        return pd.Series(0.0, index=series.index)
    return (series - series.min()) / rango

acciones = y.columns.tolist()

for accion in acciones:

    # Correlacion Lineal
    correlaciones = X.apply(lambda col: col.corr(y[accion])).abs().fillna(0)
    
    # Informacion Mutua
    mi = mutual_info_regression(X, y[accion], random_state=42)
    mi_scores = pd.Series(mi, index=X.columns)
    
    # Random Forest
    rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    rf.fit(X, y[accion])
    rf_scores = pd.Series(rf.feature_importances_, index=X.columns)

    # Normalizacion
    score_corr = normalizar(correlaciones)
    score_mi   = normalizar(mi_scores)
    score_rf   = normalizar(rf_scores)
    
    puntuacion = (score_corr + score_mi + score_rf) / 3
    
    rank = puntuacion.sort_values(ascending=False).head(10)
    
    # Visualizacion
    print(f"\nTop 10 sensores para {accion}:")
    print("-" * 55)
    print(f"{'SENSOR':<15} | {'PUNTUACION':<10} | {'CORR':<5} | {'MI':<5} | {'RF':<5}")
    print("-" * 55)
    
    for sensor, puntos in rank.items():
        c = score_corr[sensor]
        m = score_mi[sensor]
        r = score_rf[sensor]
        
        print(f"{sensor:<15} | {puntos:<10.4f} | {c:<5.2f} | {m:<5.2f} | {r:<5.2f}")

Datos sincronizados: 101029 filas analizadas.

Top 10 sensores para A_gear:
-------------------------------------------------------
SENSOR          | PUNTUACION | CORR  | MI    | RF   
-------------------------------------------------------
gear            | 1.0000     | 1.00  | 1.00  | 1.00 
speed           | 0.5588     | 0.92  | 0.75  | 0.00 
track_9         | 0.3575     | 0.60  | 0.47  | 0.00 
track_10        | 0.2675     | 0.55  | 0.25  | 0.00 
z               | 0.2315     | 0.58  | 0.12  | 0.00 
track_8         | 0.2296     | 0.44  | 0.25  | 0.00 
rpm             | 0.2250     | 0.53  | 0.13  | 0.01 
track_16        | 0.2154     | 0.38  | 0.27  | 0.00 
track_0         | 0.2112     | 0.30  | 0.33  | 0.00 
track_17        | 0.2104     | 0.39  | 0.25  | 0.00 

Top 10 sensores para A_steer:
-------------------------------------------------------
SENSOR          | PUNTUACION | CORR  | MI    | RF   
-------------------------------------------------------
track_4         | 0.9350     | 0.